# Day 3 — Amazon Fulfilment Centre Industrial Visit: Observation-to-Agent Analysis

**Industrial visit: Thursday, 20 August 2026 — Amazon fulfilment centre, UK.**

This practical is deliberately framed as **observation → evidence → candidate agent design**, not as reverse engineering Amazon systems. Students should use only information they are permitted to observe or record and must not infer confidential architecture from a tour.

>**Dr Julius Sechang Mboli**
>
>**DAIM, University of Hull**
>
>**https://www.hull.ac.uk/staff-directory/julius-mboli**
>
>**https://www.linkedin.com/in/engr-julius-sechang-mboli/**
>
>**https://jsmboli.github.io/**

In [ ]:
from pathlib import Path
import os, sys, json, time, importlib.util
import pandas as pd
pd.set_option('display.max_colwidth', None)

# Locate the package without relying on a fixed working directory.
cwd = Path.cwd().resolve()
RESOURCE_DIR = None
for p in [cwd, *cwd.parents]:
    if (p / "resources" / "src").exists():
        RESOURCE_DIR = p / "resources"
        break
    if (p / "src").exists() and (p / "notebooks").exists() and (p / "data").exists():
        RESOURCE_DIR = p
        break
if RESOURCE_DIR is None:
    raise RuntimeError("Could not locate resources/src. Extract the complete bootcamp ZIP and open this notebook from inside it.")

SRC = RESOURCE_DIR / "src"
DATA = RESOURCE_DIR / "data"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from providers import ProviderRouter, ProviderError, make_messages
from notebook_utils import (
    environment_table, provider_table, response_table,
    run_with_progress, progress_indicator, display_response, display_note,
)

router = ProviderRouter(verbose=True)
display(environment_table(RESOURCE_DIR, router))

In [2]:
from tools import retrieve_observations, risk_assessment, knowledge_lookup

In [ ]:
status = run_with_progress("Checking Groq, Ollama and optional providers", router.diagnose, True)
display(provider_table(status))

PRIMARY_PROVIDER = (
    "groq" if status["groq"]["available"]
    else "ollama" if status["ollama"]["available"]
    else "mimic"
)
print("Primary live provider for this notebook:", PRIMARY_PROVIDER)
print("Groq model:", status["groq"].get("model"))
print("Ollama model:", status["ollama"].get("model"))

## 1. Responsible observation protocol

During the visit, record process-level observations only where permitted. Do not collect names, employee identifiers, screens containing personal/customer data, security controls, access credentials, proprietary performance thresholds or photographs where photography is restricted.

For every note separate:
- **Observed:** what you directly saw/heard and are allowed to record.
- **Inferred:** your interpretation, clearly labelled as inference.
- **Agent opportunity:** a hypothetical design idea.
- **Boundary:** what the agent must not decide or do autonomously.
- **Human checkpoint:** who should review a consequential or uncertain action.

## 2. Classroom observation dataset

The supplied CSV contains synthetic/generalised examples so the notebook is runnable even before the visit. After the visit, replace/add rows only with non-sensitive observations you are permitted to use.

In [4]:
obs = pd.DataFrame(retrieve_observations("all"))
print("Observation file:", DATA / "amazon_visit_observation_examples.csv")
print("Rows:", len(obs))
display(obs)

Observation file: /Users/906845/Library/CloudStorage/OneDrive-hull.ac.uk/Documents/projects/Agentic AI Boot Camp/Agentic_AI_Bootcamp_v3_Reliable_Providers/resources/data/amazon_visit_observation_examples.csv
Rows: 6


,observation_id,area,observed_process,data_signals,possible_agentic_use_case,human_review_needed
0,OBS-001,Inbound,"Goods are received, scanned, checked, and routed for stowing.","barcode scan, arrival time, supplier data, exception notes",Inbound exception triage assistant that summarises likely causes and routes to the right staff member.,"Yes, before supplier escalation."
1,OBS-002,Stow,Items are placed into storage locations with scan confirmation.,"item ID, tote ID, pod/location, scan success, mismatch flag",Stow guidance assistant that explains mismatch errors and suggests next safe steps.,"Yes, for repeated mismatch or safety concern."
2,OBS-003,Pick,Pick tasks are allocated and completed with scanning.,"order priority, pick time, item location, error flags",Pick-support agent that answers procedural questions and flags abnormal delays.,No for basic guidance; yes for disciplinary or safety decisions.
3,OBS-004,Pack,"Orders are checked, packed, labelled, and sent to shipping.","weight, dimensions, packaging type, damage flag",Packaging recommendation agent that suggests materials and asks for confirmation if a fragile/damaged item is detected.,"Yes, before overriding damage flags."
4,OBS-005,Safety,Staff and automation operate under marked zones and safety procedures.,"near-miss reports, training status, zone alerts",Safety briefing assistant that generates role-specific reminders from approved policy.,"Always, for safety-policy changes."
5,OBS-006,Quality,"Quality checks identify mis-picks, missing items, or damaged goods.","defect type, image note, operator note, resolution",Quality-review agent that clusters defects and drafts improvement suggestions.,"Yes, before any operational decision."


## 3. Analyse observations one-by-one with progress and provenance

Each model call records which provider/model produced the analysis. The prompt forces a distinction between evidence and inference and prohibits claims of internal Amazon access.

In [5]:
from tqdm.auto import tqdm

analyses=[]
for _,row in tqdm(obs.iterrows(), total=len(obs), desc="Analysing visit observations"):
    prompt=f"""
This is a teaching exercise about a UK fulfilment-centre visit.
Observation ID: {row.get('observation_id')}
Area: {row.get('area')}
Observed process: {row.get('observed_process')}
Signals: {row.get('data_signals')}
Human checkpoint: {row.get('human_checkpoint')}

Return a concise analysis with exactly these headings:
OBSERVED EVIDENCE
INFERENCES (clearly labelled)
CANDIDATE AGENT
TOOLS OR DATA
ALLOWED ACTIONS
PROHIBITED ACTIONS
HUMAN REVIEW
FAILURE MODES
Do not claim access to Amazon internal systems or private data.
"""
    try:
        r=run_with_progress(
            f"Analyse {row.get('observation_id')}", router.chat, make_messages(prompt),
            provider=PRIMARY_PROVIDER, max_tokens=500,
            fallback_on_error=(PRIMARY_PROVIDER=="mimic")
        )
        analyses.append({
            "observation_id":row.get("observation_id"),"area":row.get("area"),
            **r.summary(),"analysis":r.text
        })
    except Exception as exc:
        analyses.append({"observation_id":row.get("observation_id"),"area":row.get("area"),"error":str(exc)})
analysis_df=pd.DataFrame(analyses)
display(analysis_df[[c for c in ["observation_id","area","provider","model","latency_s","error"] if c in analysis_df.columns]])

Analysing visit observations:   0%|          | 0/6 [00:00<?, ?it/s]

▶ Groq request started | model=qwen/qwen3.6-27b | max_tokens=500
✓ Groq completed in 1.138s | model=qwen/qwen3.6-27b


▶ Groq request started | model=qwen/qwen3.6-27b | max_tokens=500
✓ Groq completed in 1.199s | model=qwen/qwen3.6-27b


▶ Groq request started | model=qwen/qwen3.6-27b | max_tokens=500
✓ Groq completed in 0.872s | model=qwen/qwen3.6-27b


▶ Groq request started | model=qwen/qwen3.6-27b | max_tokens=500
✓ Groq completed in 1.301s | model=qwen/qwen3.6-27b


▶ Groq request started | model=qwen/qwen3.6-27b | max_tokens=500
✓ Groq completed in 1.227s | model=qwen/qwen3.6-27b


▶ Groq request started | model=qwen/qwen3.6-27b | max_tokens=500
✓ Groq completed in 1.151s | model=qwen/qwen3.6-27b


,observation_id,area,provider,model,latency_s
0,OBS-001,Inbound,groq,qwen/qwen3.6-27b,1.138
1,OBS-002,Stow,groq,qwen/qwen3.6-27b,1.199
2,OBS-003,Pick,groq,qwen/qwen3.6-27b,0.872
3,OBS-004,Pack,groq,qwen/qwen3.6-27b,1.301
4,OBS-005,Safety,groq,qwen/qwen3.6-27b,1.227
5,OBS-006,Quality,groq,qwen/qwen3.6-27b,1.151


## 4. Student note triage

Replace the example notes below with permitted notes from the visit. The goal is not to label “good” or “bad” workers; it is to classify whether a note is suitable evidence for an agent-design exercise and whether it contains sensitive or consequential content that should be excluded or escalated.

In [6]:
student_notes=[
    "Operators encounter exceptions where an item or tote needs additional checking.",
    "A process includes safety-critical decisions around an unusual condition.",
    "A screen showed a person's name and an individual performance value.",
    "Items move through several stages with scan confirmations and exception handling."
]
rows=[]
for note in student_notes:
    risk=risk_assessment(note)
    rows.append({"note":note, **risk,
                 "teaching_use":"exclude personal identifiers" if "name" in note.lower() else "retain as generalised process observation"})
display(pd.DataFrame(rows))

,note,risk,signals,recommendation,teaching_use
0,Operators encounter exceptions where an item or tote needs additional checking.,medium,none,continue_with_logging,retain as generalised process observation
1,A process includes safety-critical decisions around an unusual condition.,high,safety,human_review,retain as generalised process observation
2,A screen showed a person's name and an individual performance value.,high,performance,human_review,exclude personal identifiers
3,Items move through several stages with scan confirmations and exception handling.,medium,none,continue_with_logging,retain as generalised process observation


## 5. Post-visit synthesis challenge

In teams, select **three** permitted observations and design one agentic workflow. Your submission must include:
1. user and goal;
2. graph state;
3. deterministic tools;
4. one LLM decision;
5. evidence provenance;
6. an explicit action boundary;
7. at least one LangGraph `interrupt()` checkpoint;
8. failure and recovery behaviour;
9. provider/model metadata to record;
10. a statement distinguishing observed facts from your proposed design.

The next two notebooks turn this evidence into RAG and a capstone agent.